Tools

In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Files

In [ ]:
from google.colab import files
files.upload()

**Reading the files**

In [ ]:
df = pd.read_csv("air_quality_historical.csv")
city = pd.read_csv("city_info.csv")

print(df.shape)
df.head()

# ***Data Cleaning***

Making the date work as a date

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")
df = df.set_index("date")

df.head()

**Counting the missing values**

In [ ]:
df.isnull().sum()

**Removing the empty days and checking days**

In [ ]:
df = df.dropna(subset=["pm2_5"])

all_days = pd.date_range(df.index.min(), df.index.max(), freq="D")
print("Days there should be:", len(all_days))
print("Days in my data:", len(df))
print("Duplicate dates:", df.index.duplicated().sum())

**Looking Numbers**

In [ ]:
df = df.asfreq("D")
df.describe().round(2)

# ***Drawing charts to understand PM2.5***

**PM2.5 over time**

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df.index, df["pm2_5"], color="lightblue", label="daily")
plt.plot(df.index, df["pm2_5"].rolling(30).mean(), color="navy", label="30 day average")
plt.axhline(15, color="red", linestyle="--", label="WHO daily guideline (15)")
plt.ylabel("PM2.5 (ug/m3)")
plt.title("PM2.5 in Lahore")
plt.legend(loc="upper left")
plt.show()

**Average PM2.5 in each month**

In [ ]:
month_avg = df.groupby(df.index.month)["pm2_5"].mean()

plt.figure(figsize=(8, 4))
plt.bar(month_avg.index, month_avg.values, color="steelblue")
plt.axhline(15, color="red", linestyle="--")
plt.xticks(range(1, 13))
plt.xlabel("Month")
plt.ylabel("Average PM2.5")
plt.title("Average PM2.5 in each month")
plt.show()

month_avg.round(1)

**Smog season and the other months**

In [ ]:
smog_months = [10, 11, 12, 1]
is_smog = df.index.month.isin(smog_months)

print("Average PM2.5 in smog season (Oct-Jan):", round(df[is_smog]["pm2_5"].mean(), 1))
print("Average PM2.5 in the other months     :", round(df[~is_smog]["pm2_5"].mean(), 1))
print("Percent of days above 15:", round((df["pm2_5"] > 15).mean() * 100, 1), "%")

**My observation**: PM2.5 is highest in January (about 140) and December (about 117), and lowest in April (about 38). The smog season (Oct to Jan) has an average of about 109, but the other months have an average of about 51, so it is about 2 times higher in the smog season. The pattern repeats every year. Even in the good months, almost every day (99.7%) is above the WHO guideline of 15.

# ***Downloading the weather data***

**Downloading the weather and adding in my table**

In [ ]:
import requests

lat = float(city.loc[0, "latitude"])
lon = float(city.loc[0, "longitude"])
print("Latitude:", lat, "| Longitude:", lon)

url = "https://archive-api.open-meteo.com/v1/archive"
settings = {
    "latitude": lat,
    "longitude": lon,
    "start_date": df.index.min().strftime("%Y-%m-%d"),
    "end_date": df.index.max().strftime("%Y-%m-%d"),
    "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation",
    "timezone": "Asia/Karachi",
}

weather_ok = False
try:
    reply = requests.get(url, params=settings, timeout=90)
    reply.raise_for_status()
    hourly = pd.DataFrame(reply.json()["hourly"])
    hourly["time"] = pd.to_datetime(hourly["time"])
    hourly = hourly.set_index("time")

    weather = pd.DataFrame()
    weather["temp"] = hourly["temperature_2m"].resample("D").mean()
    weather["humidity"] = hourly["relative_humidity_2m"].resample("D").mean()
    weather["wind"] = hourly["wind_speed_10m"].resample("D").mean()
    weather["rain"] = hourly["precipitation"].resample("D").sum()

    df = df.join(weather)
    weather_ok = True
    print("Weather downloaded.")
except Exception as e:
    print("Weather download failed, I will continue without weather.")
    print("Error:", e)

**Checking the new columns**

In [ ]:
df[["pm2_5", "temp", "humidity", "wind", "rain"]].head()

In [ ]:
df[["temp", "humidity", "wind", "rain"]].isnull().sum()

**Does weather go together with PM2.5?**

In [ ]:
if weather_ok:
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.scatter(df["temp"], df["pm2_5"], s=6, alpha=0.4)
    plt.xlabel("Temperature")
    plt.ylabel("PM2.5")

    plt.subplot(1, 3, 2)
    plt.scatter(df["humidity"], df["pm2_5"], s=6, alpha=0.4)
    plt.xlabel("Humidity")

    plt.subplot(1, 3, 3)
    plt.scatter(df["wind"], df["pm2_5"], s=6, alpha=0.4)
    plt.xlabel("Wind speed")

    plt.tight_layout()
    plt.show()

    print(df[["pm2_5", "temp", "humidity", "wind", "rain"]].corr()["pm2_5"].round(2))
else:
    print("No weather data, skipping this step.")

# ***Preparing the data for prediction***

**Making a copy of my data and creating the target**

In [ ]:
# Making a copy of my data
data = df.copy()

# Creating the target: tomorrow's PM2.5
data["tomorrow_pm25"] = data["pm2_5"].shift(-1)

data[["pm2_5", "tomorrow_pm25"]].head()

**Creating columns with older PM2.5 values**

In [ ]:
# Creating columns with older PM2.5 values
data["pm25_yesterday"] = data["pm2_5"].shift(1)
data["pm25_week_ago"] = data["pm2_5"].shift(7)
data["pm25_avg7"] = data["pm2_5"].rolling(7).mean()

data[["pm2_5", "pm25_yesterday", "pm25_week_ago", "pm25_avg7"]].head(10)

**Creating the month columns**

In [ ]:
# Creating the month columns
data["month_sin"] = np.sin(2 * np.pi * data.index.month / 12)
data["month_cos"] = np.cos(2 * np.pi * data.index.month / 12)

data[["month_sin", "month_cos"]].head(3)

**Choosing my input columns and removing the incomplete rows**

In [ ]:
# Choosing my input columns
pollution_cols = ["pm2_5", "pm10", "nitrogen_dioxide", "sulphur_dioxide", "carbon_monoxide", "ozone",
                  "pm25_yesterday", "pm25_week_ago", "pm25_avg7", "month_sin", "month_cos"]

weather_cols = []
if weather_ok:
    weather_cols = ["temp", "humidity", "wind", "rain"]

all_cols = pollution_cols + weather_cols

# Removing the rows that have empty values
data = data.dropna(subset=all_cols + ["tomorrow_pm25"])

print("Rows I can use:", len(data))
print("Number of input columns:", len(all_cols))

**Splitting the data into train and test**

In [ ]:
# Splitting the data into train and test (by date)
train = data[data.index < "2025-03-01"]
test = data[data.index >= "2025-03-01"]

X_train = train[all_cols]
y_train = train["tomorrow_pm25"]
X_test = test[all_cols]
y_test = test["tomorrow_pm25"]

print("Train days:", len(train), "| from", train.index.min().date(), "to", train.index.max().date())
print("Test days :", len(test), "| from", test.index.min().date(), "to", test.index.max().date())

# ***Training the models and comparing them***

**Importing the model tools**

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

**Making a function to get the scores**

In [ ]:
all_results = []

def get_scores(name, y_real, y_pred):
    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2 = r2_score(y_real, y_pred)
    all_results.append({"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2})

**Making the two simple guesses**

In [ ]:
guess_today = X_test["pm2_5"]
get_scores("Guess: tomorrow = today", y_test, guess_today)

train_month_avg = train.groupby((train.index + pd.Timedelta(days=1)).month)["tomorrow_pm25"].mean()
test_months = (X_test.index + pd.Timedelta(days=1)).month
guess_month = [train_month_avg[m] for m in test_months]
get_scores("Guess: month average", y_test, guess_month)

print("Simple guesses done")

**Training the 3 models**

In [ ]:
# Linear Regression
lin_model = LinearRegression()
lin_model.fit(X_train, y_train)
get_scores("Linear Regression", y_test, lin_model.predict(X_test))

# Random Forest
rf_model = RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_leaf=10, random_state=42)
rf_model.fit(X_train, y_train)
get_scores("Random Forest", y_test, rf_model.predict(X_test))

# Gradient Boosting
gb_model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.03, max_depth=3, random_state=42)
gb_model.fit(X_train, y_train)
get_scores("Gradient Boosting", y_test, gb_model.predict(X_test))

print("All 3 models trained")

**Comparing all the results**

In [ ]:

results = pd.DataFrame(all_results).sort_values("MAE").reset_index(drop=True)
results.round(2)

**Drawing a chart of the MAE**

In [ ]:
plt.figure(figsize=(8, 4))
plt.barh(results["Model"], results["MAE"], color="steelblue")
plt.xlabel("MAE on test data (lower is better)")
plt.title("Model comparison")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# ***Checking where my model makes mistakes***

**Finding the best model**

In [ ]:
models = {"Linear Regression": lin_model, "Random Forest": rf_model, "Gradient Boosting": gb_model}

best_name = None
for name in results["Model"]:
    if name in models:
        best_name = name
        break

best_model = models[best_name]
print("Best model:", best_name)

**Making a table of actual and predicted values**

In [ ]:
check = pd.DataFrame()
check["actual"] = y_test.values
check["predicted"] = best_model.predict(X_test)
check["guess_today"] = X_test["pm2_5"].values
check.index = X_test.index + pd.Timedelta(days=1)

check["model_error"] = (check["actual"] - check["predicted"]).abs()
check["today_error"] = (check["actual"] - check["guess_today"]).abs()

check.head()

**Drawing actual vs predicted**

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(check.index, check["actual"], color="black", label="Actual")
plt.plot(check.index, check["predicted"], color="red", label="Predicted")
plt.ylabel("PM2.5")
plt.title("Actual vs predicted on the test days")
plt.legend(loc="upper left")
plt.show()

**Is the model worse in the smog season?**

In [ ]:
smog_months = [10, 11, 12, 1]
check["smog_season"] = check.index.month.isin(smog_months)

print("Average mistake in smog season :", round(check[check["smog_season"]]["model_error"].mean(), 1))
print("Average mistake in other months:", round(check[~check["smog_season"]]["model_error"].mean(), 1))
print()
print("Average mistake when actual PM2.5 is above 100  :", round(check[check["actual"] > 100]["model_error"].mean(), 1))
print("Average mistake when actual PM2.5 is 100 or less:", round(check[check["actual"] <= 100]["model_error"].mean(), 1))

**Looking at the 10 biggest mistakes**

In [ ]:
check.sort_values("model_error", ascending=False).head(10)[["actual", "predicted", "model_error"]].round(1)

**Checking sudden changes**

In [ ]:
check["change"] = (check["actual"] - check["guess_today"]).abs()
big_change = check[check["change"] > 40]

print("Days with a change of more than 40:", len(big_change))
print("Average mistake of my model on those days:", round(big_change["model_error"].mean(), 1))

# ***Finding which inputs matter most***

**Getting the importance of each input**

In [ ]:
importance = pd.Series(rf_model.feature_importances_, index=all_cols)
importance = importance.sort_values(ascending=False)

importance.round(3)

**Drawing a chart of the top 10 inputs**

In [ ]:
top = importance.head(10).sort_values()

plt.figure(figsize=(8, 5))
plt.barh(top.index, top.values, color="steelblue")
plt.xlabel("Importance")
plt.title("Top 10 inputs (Random Forest)")
plt.tight_layout()
plt.show()

# ***Testing if weather really helps***

**Testing the Random Forest with and without weather**

In [ ]:
if weather_ok:
    without_weather = []
    with_weather = []

    for seed in [1, 2, 3]:
        # Training without weather
        m1 = RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_leaf=10, random_state=seed)
        m1.fit(train[pollution_cols], y_train)
        without_weather.append(mean_absolute_error(y_test, m1.predict(test[pollution_cols])))

        # Training with weather
        m2 = RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_leaf=10, random_state=seed)
        m2.fit(train[all_cols], y_train)
        with_weather.append(mean_absolute_error(y_test, m2.predict(test[all_cols])))

    print("MAE without weather (3 runs):", np.round(without_weather, 2))
    print("MAE with weather    (3 runs):", np.round(with_weather, 2))
    print()
    print("Average MAE without weather:", round(np.mean(without_weather), 2))
    print("Average MAE with weather   :", round(np.mean(with_weather), 2))

    change = (np.mean(with_weather) - np.mean(without_weather)) / np.mean(without_weather) * 100
    print("Change:", round(change, 1), "%   (negative means weather helped)")
else:
    print("No weather data, so I cannot do this test.")

**Testing the Linear Regression with and without weather**

In [ ]:
if weather_ok:
    lin1 = LinearRegression()
    lin1.fit(train[pollution_cols], y_train)
    mae_without = mean_absolute_error(y_test, lin1.predict(test[pollution_cols]))

    lin2 = LinearRegression()
    lin2.fit(train[all_cols], y_train)
    mae_with = mean_absolute_error(y_test, lin2.predict(test[all_cols]))

    print("Linear Regression MAE without weather:", round(mae_without, 2))
    print("Linear Regression MAE with weather   :", round(mae_with, 2))
    print("Change:", round((mae_with - mae_without) / mae_without * 100, 1), "%")
else:
    print("No weather data, so I cannot do this test.")

# ***Getting my key numbers and writing my conclusion***

**Getting my key numbers**

In [ ]:
today_mae = results[results["Model"] == "Guess: tomorrow = today"]["MAE"].iloc[0]
month_mae = results[results["Model"] == "Guess: month average"]["MAE"].iloc[0]
best_mae = results[results["Model"] == best_name]["MAE"].iloc[0]
best_r2 = results[results["Model"] == best_name]["R2"].iloc[0]

print("Best model:", best_name)
print("Best model MAE:", round(best_mae, 1), "| R2:", round(best_r2, 2))
print("MAE of 'tomorrow = today':", round(today_mae, 1))
print("MAE of 'month average'   :", round(month_mae, 1))
print("Better than 'tomorrow = today' by:", round((today_mae - best_mae) / today_mae * 100, 1), "%")
print("Test period:", test.index.min().date(), "to", test.index.max().date(), "(", len(test), "days )")

**Introduction**

**Name:** Muhammad Ahmad

## What is this project about?
Lahore has very bad air pollution, especially in winter (smog). I used daily air quality
data to predict tomorrow's PM2.5 from today's data. I also checked if adding weather
data makes the prediction better.

## My questions
1. Is PM2.5 higher in some months than in others?
2. Can a model predict tomorrow's PM2.5 better than a simple guess?
3. Does weather data help?
4. Where does the model make big mistakes?

## Data
- Air quality: Kaggle dataset "Air Quality Dataset for Lahore" by Nitiraj Kulkarni (Aug 2022 to Feb 2026).
- This data comes from a computer model (CAMS), so it is NOT from real ground monitors.
- Weather: downloaded from Open-Meteo.

# **Conclusion**

## Conclusion

**What I found**
- PM2.5 is highest in January and lowest in April. The smog season (Oct-Jan) average was 108.9, and the other months were 50.8.
- The best model was Linear Regression with MAE 12.9. The simple guess "tomorrow = today" had MAE 14.1, so my model was better by only 8.6%.
- The model mostly follows today's PM2.5. Today's PM2.5 was the most important input (91% in the Random Forest).
- Weather did not clearly help. The MAE of the Random Forest changed by -0.7% (from 13.22 to 13.12), and the MAE of the Linear Regression changed by +0.1% (from 12.88 to 12.89). These changes are very small, so weather did not make a clear difference in my data. One possible reason is that today’s PM2.5 already contains a lot of the weather information, because smog and weather change together.
- The biggest mistakes happened on 2025-05-22, when the real PM2.5 was about 152 but my model predicted about 45. The model was weak on sudden jumps and drops: on the 20 days with a change of more than 40, the average mistake was about 43, compared with about 13 overall.

**Limitations**
- The data is from a computer model (CAMS), not from ground monitors.
- It is one city and about 3.5 years. My test period has only one smog season.
- I did not tune the model settings.
- I picked the best model using the test data, which is a little optimistic.
- The daily average hides hourly peaks.
- I only predict 1 day ahead, and I do not use data about pollution sources (traffic, burning, industry).
- I tested weather on only one test period (one smog season), so the result could be different with other data.


**What I can do next**
- Check my results with real monitor data (for example from OpenAQ).
- Predict 2 or 3 days ahead.
- Try other cities like Karachi and Islamabad.

## Credits
Dataset by Nitiraj Kulkarni (Kaggle). Weather data from Open-Meteo.